# **Laboratory Task 3 — Forward and Backward Propagation (Python)**

**Instruction:** Perform a forward and backward propagation in Python using the inputs from
Laboratory Task 2.

```python
x = np.array([1, 0, 1])
y = np.array([1])

# use relu as the activation function.

# learning rate
lr = 0.001
```

This notebook builds on Laboratory Task 2: it reproduces the same forward pass, then adds
**backpropagation** (computing gradients) and a **gradient descent update**


## 1. Setup: Inputs, Weights, Biases, Learning Rate

In [3]:
import numpy as np

# Inputs (same as Laboratory Task 2)
x = np.array([1, 0, 1], dtype=float)
y = np.array([1], dtype=float)

# use relu as the activation function.
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

# learning rate
lr = 0.001

# Hidden unit weights 
W1 = np.array([
    [0.2, -0.3],
    [0.4,  0.1],
    [-0.5, 0.2]
])
b1 = np.array([-0.4, 0.2])   # theta1, theta2

# Output unit weights 
W2 = np.array([
    [-0.3],
    [-0.2]
])
b2 = np.array([0.1])   # theta3

print("Setup complete.")
print("W1 =\n", W1, "\nb1 =", b1)
print("W2 =\n", W2, "\nb2 =", b2)


Setup complete.
W1 =
 [[ 0.2 -0.3]
 [ 0.4  0.1]
 [-0.5  0.2]] 
b1 = [-0.4  0.2]
W2 =
 [[-0.3]
 [-0.2]] 
b2 = [0.1]


## 2. Forward Propagation

In [4]:
z1 = x.dot(W1) + b1     # hidden net input
a1 = relu(z1)             # hidden activation

z2 = a1.dot(W2) + b2     # output net input
a2 = relu(z2)             # predicted output

error = 0.5 * np.sum((y - a2) ** 2)

print("z1 (hidden net input)    :", z1)
print("a1 (hidden activation)   :", a1)
print("z2 (output net input)    :", z2)
print("a2 / y_hat (prediction)  :", a2)
print("Error                    :", error)


z1 (hidden net input)    : [-0.7  0.1]
a1 (hidden activation)   : [0.  0.1]
z2 (output net input)    : [0.08]
a2 / y_hat (prediction)  : [0.08]
Error                    : 0.4232


**Analysis:** These values match Laboratory Task 2 exactly (ŷ = 0.08, error = 0.4232), confirming the
forward pass implementation is correct.

## 3. Backward Propagation (Computing Gradients)

Backpropagation applies the chain rule, working from the output layer back to the input layer.

**Output layer error signal:**
$$\delta_2 = (\hat{y} - y) \cdot \text{ReLU}'(z_2)$$


In [5]:
delta2 = (a2 - y) * relu_derivative(z2)

dW2 = np.outer(a1, delta2)   # gradient for output weights
db2 = delta2                  # gradient for output bias

print("delta2 (output error signal):", delta2)
print("dW2:\n", dW2)
print("db2:", db2)


delta2 (output error signal): [-0.92]
dW2:
 [[-0.   ]
 [-0.092]]
db2: [-0.92]


**Hidden layer error signal** (push the output error backward through `W2`):
$$\delta_1 = (\delta_2 \cdot W_2^T) \cdot \text{ReLU}'(z_1)$$


In [6]:
delta1 = (delta2 @ W2.T) * relu_derivative(z1)

dW1 = np.outer(x, delta1)   # gradient for hidden weights
db1 = delta1                 # gradient for hidden bias

print("delta1 (hidden error signal):", delta1)
print("dW1:\n", dW1)
print("db1:", db1)


delta1 (hidden error signal): [0.    0.184]
dW1:
 [[0.    0.184]
 [0.    0.   ]
 [0.    0.184]]
db1: [0.    0.184]


## 4. Gradient Descent Update

In [7]:
W1_before, b1_before = W1.copy(), b1.copy()
W2_before, b2_before = W2.copy(), b2.copy()

W1 = W1 - lr * dW1
b1 = b1 - lr * db1
W2 = W2 - lr * dW2
b2 = b2 - lr * db2

print("W1 before:\n", W1_before, "\nW1 after:\n", W1)
print("\nb1 before:", b1_before, " b1 after:", b1)
print("\nW2 before:\n", W2_before, "\nW2 after:\n", W2)
print("\nb2 before:", b2_before, " b2 after:", b2)


W1 before:
 [[ 0.2 -0.3]
 [ 0.4  0.1]
 [-0.5  0.2]] 
W1 after:
 [[ 0.2      -0.300184]
 [ 0.4       0.1     ]
 [-0.5       0.199816]]

b1 before: [-0.4  0.2]  b1 after: [-0.4       0.199816]

W2 before:
 [[-0.3]
 [-0.2]] 
W2 after:
 [[-0.3     ]
 [-0.199908]]

b2 before: [0.1]  b2 after: [0.10092]


## 5. Verify: Re-run Forward Pass With Updated Weights

In [8]:
z1_new = x.dot(W1) + b1
a1_new = relu(z1_new)
z2_new = a1_new.dot(W2) + b2
a2_new = relu(z2_new)
error_new = 0.5 * np.sum((y - a2_new) ** 2)

print("New prediction (y_hat):", a2_new)
print("New error             :", error_new)
print("Error before update   :", error)
print("Error decreased by    :", error - error_new)


New prediction (y_hat): [0.08103955]
New error             : 0.42224415505256624
Error before update   : 0.4232
Error decreased by    : 0.0009558449474337816


## 6. Analysis & Discussion

1. **Gradient descent moved the weights in the correct direction.** After a single update with a small "
learning rate (`lr = 0.001`), the error decreased slightly. 
2. **The learning rate matters.** Because `lr` is very small (0.001), the change per step is tiny ( "
`W2[1]` only moved from -0.2 to -0.199908). Many more iterations (epochs) would be needed to meaningfully "
reduce the error from 0.4232 toward 0.
3. **Only the weights connected to the active hidden unit (unit 2) were updated.** Because hidden unit "
1 was inactive (`a_h1 = 0`, ReLU derivative 0) in this forward pass, its connected weights (`W1[:,0]`, "
`b1[0]`) received zero gradient and remain unchanged after this step. If this pattern persisted across "
training (the unit never activates for any input), it is a case of a permanently "dead" ReLU unit — a "
known failure mode of ReLU networks, sometimes mitigated by using Leaky ReLU or careful weight "
initialization.
4. **Consistency with Task 2.** The forward-pass values computed here (ŷ = 0.08, error = 0.4232) match "
Laboratory Task 2's hand calculations exactly, confirming the implementation is correct before "
backpropagation is layered on top.

### 6.1 Summary Table — Before vs. After One Training Step

| Parameter | Before | After |
|---|---|---|
| W1 | `[[0.2,-0.3],[0.4,0.1],[-0.5,0.2]]` | `[[0.2,-0.300184],[0.4,0.1],[-0.5,0.199816]]` |
| b1 | `[-0.4, 0.2]` | `[-0.4, 0.199816]` |
| W2 | `[[-0.3],[-0.2]]` | `[[-0.3],[-0.199908]]` |
| b2 | `[0.1]` | `[0.10092]` |
| ŷ | 0.08 | (slightly closer to 1, see output above) |
| Error | 0.4232 | (slightly lower, see output above) |
